# ATLAS: bounded Gemma 4 E4B runtime probe

This notebook tests first-token generation on the selected Kaggle hardware. It does not measure agent repair performance. One attempt, one model variant, FP16, tensor parallelism 1, 2,048-token context, 32 generated tokens. Installation and inference have external time limits. The official competition wheelhouse and pinned starter image supply the runtime.

Sources: [official starter](https://www.kaggle.com/code/ryanholbrook/getting-started-gemma-4-developer-agent), [Paper Track](https://www.kaggle.com/competitions/gemma-4-developer-agent-paper).

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time
WORK = Path('/kaggle/working')
WORK.mkdir(exist_ok=True)
START = time.monotonic()
(WORK / 'install_runtime.py').write_text('import glob\nimport importlib\nimport logging\nimport os\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\n# Configure environment variables for offline vLLM serving and LiteLLM routing\nos.environ[\'TRANSFORMERS_NO_TF\'] = \'1\'\nos.environ[\'TRANSFORMERS_NO_FLAX\'] = \'1\'\nos.environ[\'TRANSFORMERS_OFFLINE\'] = \'1\'\nos.environ[\'VLLM_NO_USAGE_STATS\'] = \'1\'\nos.environ[\'VLLM_WORKER_MULTIPROC_METHOD\'] = \'spawn\'\nos.environ[\'VLLM_ENGINE_READY_TIMEOUT_S\'] = \'1200\'\nos.environ[\'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS\'] = \'1\'\nos.environ[\'NCCL_DEBUG\'] = \'WARN\'\nos.environ[\'LITELLM_LOCAL_MODEL_COST_MAP\'] = \'True\'\nos.environ[\'LITELLM_TELEMETRY\'] = \'False\'\nos.environ[\'LITELLM_LOG\'] = \'WARNING\'\nos.environ[\'OPENAI_API_KEY\'] = os.environ.get(\'OPENAI_API_KEY\', \'EMPTY\')\nos.environ[\'LOCAL_API_KEY\'] = os.environ.get(\'LOCAL_API_KEY\', \'EMPTY\')\nos.environ[\'PYTORCH_CUDA_ALLOC_CONF\'] = \'expandable_segments:True\'\nos.environ[\'OTEL_SDK_DISABLED\'] = \'true\'\nos.environ[\'OTEL_PYTHON_DISABLED\'] = \'true\'\n\n\ndef configure_logging() -> None:\n    """Silence verbose third-party loggers."""\n    for name in (\n        \'LiteLLM\',\n        \'LiteLLM Router\',\n        \'LiteLLM Proxy\',\n        \'litellm\',\n        \'httpx\',\n        \'httpcore\',\n        \'openai\',\n        \'urllib3\',\n        \'opentelemetry\',\n        \'opentelemetry.context\',\n    ):\n        lg = logging.getLogger(name)\n        lg.setLevel(logging.WARNING)\n        if \'litellm\' in name.lower():\n            lg.propagate = False\n            lg.handlers.clear()\n\n\nconfigure_logging()\n\nWHEELHOUSE_DIR = Path(\'/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse\')\n\n# Remove broken cutlass .pth hooks if present\nfor pth_pattern in (\n    \'/usr/local/lib/python*/dist-packages/*cutlass*.pth\',\n    \'/usr/local/lib/python*/site-packages/*cutlass*.pth\',\n):\n    for pth in glob.glob(pth_pattern):\n        try:\n            os.unlink(pth)\n        except OSError:\n            pass\n\n# Restore PEP 440 \'+cu128\' wheel filenames stripped by Kaggle dataset uploads\ntmp_whl = Path(\'/tmp/wheelhouse\')\ntmp_whl.mkdir(parents=True, exist_ok=True)\nfor w in WHEELHOUSE_DIR.glob(\'*.whl\'):\n    if \'cutlass\' in w.name.lower():\n        continue\n    target_name = (\n        w.name.replace(\'cu128\', \'+cu128\')\n        if (\'cu128\' in w.name and \'+\' not in w.name)\n        else w.name\n    )\n    target = tmp_whl / target_name\n    if not target.exists():\n        try:\n            os.symlink(w, target)\n        except OSError:\n            shutil.copyfile(w, target)\n\nwheels = sorted(str(w) for w in tmp_whl.glob(\'*.whl\'))\nprint(f\'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...\')\nsubprocess.run(\n    [sys.executable, \'-m\', \'pip\', \'install\', \'-q\', \'--no-deps\', \'--force-reinstall\', *wheels],\n    check=True,\n)\nimportlib.invalidate_caches()\nconfigure_logging()\nprint(\'Wheelhouse installation complete.\')', encoding='utf-8')
with (WORK / 'install.log').open('w') as log:
    try:
        process = subprocess.run([sys.executable, str(WORK / 'install_runtime.py')], stdout=log, stderr=subprocess.STDOUT, timeout=420)
        installation = {'status': 'passed' if process.returncode == 0 else 'failed', 'exit_code': process.returncode}
    except subprocess.TimeoutExpired:
        installation = {'status': 'timeout', 'limit_seconds': 420}
(WORK / 'installation.json').write_text(json.dumps(installation, indent=2))
print(installation)
print((WORK / 'install.log').read_text(errors='replace')[-3000:])

In [ ]:
(WORK / 'kaggle_preflight.py').write_text('"""Bounded E4B runtime probe. No model is loaded unless --generate is supplied.\n\nThe probe establishes generation compatibility only, never an agent score.\nRun in a subprocess with an external timeout so failed GPU startup is bounded.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport importlib.metadata\nimport json\nimport os\nimport platform\nimport sys\nimport time\nfrom pathlib import Path\n\nMODEL_NAME = \'gemma-4-e4b-it-qat-w4a16-ct\'\n\n\ndef model_config(model_path: Path) -> dict:\n    model_path = Path(model_path)\n    if not (model_path / \'config.json\').is_file():\n        raise FileNotFoundError(f\'Missing model config: {model_path / "config.json"}\')\n    return {\n        \'model\': str(model_path), \'dtype\': \'half\',\n        \'tensor_parallel_size\': 1, \'max_model_len\': 2048,\n        \'gpu_memory_utilization\': 0.80, \'enforce_eager\': True,\n        \'disable_log_stats\': True,\n        \'limit_mm_per_prompt\': {\'image\': 0, \'video\': 0, \'audio\': 0},\n    }\n\n\ndef compatibility_gate(gpus: list[dict]) -> dict:\n    if not gpus:\n        return {\'status\': \'no_cuda_gpu\', \'attention_risk\': None}\n    turing = any(tuple(gpu[\'capability\']) < (8, 0) for gpu in gpus)\n    return {\n        \'status\': \'requires_runtime_test\', \'attention_risk\': turing,\n        \'note\': \'Memory capacity alone does not establish attention-kernel support.\',\n    }\n\n\ndef run_preflight(model_path: Path | None = None, *, generate: bool = False) -> dict:\n    started = time.monotonic()\n    result = {\n        \'schema_version\': 1, \'model\': MODEL_NAME,\n        \'python\': platform.python_version(), \'platform\': platform.platform(),\n        \'generation_status\': \'not_requested\', \'agent_status\': \'not_run\',\n        \'configuration_scope\': \'short-context runtime probe, not official scoring\',\n        \'packages\': {}, \'gpus\': [],\n    }\n    for package in (\'torch\', \'vllm\', \'transformers\', \'compressed-tensors\', \'swegemma\', \'adk-submission\'):\n        try:\n            result[\'packages\'][package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            result[\'packages\'][package] = None\n    try:\n        import torch\n        if torch.cuda.is_available():\n            result[\'gpus\'] = [\n                {\'name\': torch.cuda.get_device_name(i),\n                 \'capability\': list(torch.cuda.get_device_capability(i)),\n                 \'total_memory_bytes\': torch.cuda.get_device_properties(i).total_memory}\n                for i in range(torch.cuda.device_count())\n            ]\n    except ImportError:\n        pass\n    result[\'compatibility\'] = compatibility_gate(result[\'gpus\'])\n    if generate:\n        result[\'generation_status\'] = \'failed\'\n        try:\n            if not result[\'gpus\']:\n                raise RuntimeError(\'No CUDA GPU is available; no model loading attempted.\')\n            if model_path is None:\n                raise FileNotFoundError(\'Provide an attached E4B checkpoint path.\')\n            config = model_config(model_path)\n            checkpoint = json.loads((Path(model_path) / \'config.json\').read_text(encoding=\'utf-8\'))\n            result[\'checkpoint_architectures\'] = checkpoint.get(\'architectures\')\n            text = checkpoint.get(\'text_config\', checkpoint)\n            result[\'checkpoint_attention\'] = {k: text.get(k) for k in (\'head_dim\', \'global_head_dim\', \'dtype\', \'torch_dtype\')}\n            result[\'runtime_config\'] = config\n            os.environ.setdefault(\'VLLM_WORKER_MULTIPROC_METHOD\', \'spawn\')\n            os.environ.setdefault(\'VLLM_NO_USAGE_STATS\', \'1\')\n            os.environ.setdefault(\'HF_HUB_OFFLINE\', \'1\')\n            os.environ.setdefault(\'TRANSFORMERS_OFFLINE\', \'1\')\n            from vllm import LLM, SamplingParams\n            llm = LLM(**config)\n            outputs = llm.generate([\'Write a Python function add(a, b) that returns their sum.\'],\n                                   SamplingParams(max_tokens=32, temperature=0))\n            text_output = outputs[0].outputs[0].text\n            token_count = len(outputs[0].outputs[0].token_ids)\n            if not token_count:\n                raise RuntimeError(\'The runtime returned zero generated tokens.\')\n            result.update(generation_status=\'passed\', generated_text=text_output,\n                          generated_tokens=token_count)\n        except Exception as error:\n            result[\'error_type\'] = type(error).__name__\n            result[\'error\'] = str(error)[:1200]\n    result[\'elapsed_seconds\'] = round(time.monotonic() - started, 3)\n    return result\n\n\ndef main(argv=None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--model-path\', type=Path)\n    parser.add_argument(\'--generate\', action=\'store_true\')\n    parser.add_argument(\'--output\', type=Path, required=True)\n    args = parser.parse_args(argv)\n    result = run_preflight(args.model_path, generate=args.generate)\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    args.output.write_text(json.dumps(result, ensure_ascii=False, indent=2) + \'\\n\', encoding=\'utf-8\')\n    print(json.dumps(result, ensure_ascii=False, indent=2))\n    return int(args.generate and result[\'generation_status\'] != \'passed\')\n\n\nif __name__ == \'__main__\':\n    sys.exit(main())\n', encoding='utf-8')
MODEL = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-e4b-it-qat-w4a16-ct/2')
if installation['status'] != 'passed':
    result = {'generation_status': 'not_run', 'agent_status': 'not_run', 'reason': 'runtime_installation_failed'}
else:
    env = os.environ.copy()
    env.update(CUDA_VISIBLE_DEVICES='0', VLLM_WORKER_MULTIPROC_METHOD='spawn', TRANSFORMERS_NO_TF='1', TRANSFORMERS_NO_FLAX='1', VLLM_NO_USAGE_STATS='1', HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
    with (WORK / 'generation.log').open('w') as log:
        try:
            process = subprocess.run([sys.executable, str(WORK / 'kaggle_preflight.py'), '--model-path', str(MODEL), '--generate', '--output', str(WORK / 'model_preflight.json')], env=env, stdout=log, stderr=subprocess.STDOUT, timeout=600, start_new_session=True)
            if (WORK / 'model_preflight.json').exists():
                result = json.loads((WORK / 'model_preflight.json').read_text())
            else:
                result = {'generation_status': 'failed', 'agent_status': 'not_run', 'exit_code': process.returncode, 'reason': 'worker_exited_without_result'}
        except subprocess.TimeoutExpired:
            result = {'generation_status': 'timeout', 'agent_status': 'not_run', 'limit_seconds': 600}
    print((WORK / 'generation.log').read_text(errors='replace')[-5000:])
result['notebook_elapsed_seconds'] = round(time.monotonic() - START, 3)
(WORK / 'model_preflight.json').write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))

The JSON and logs in notebook outputs are the evidence. A successful short generation permits a subsequent agent smoke experiment; it does not certify a long context, all tool calls, or an official resolved score. A failed run closes this bounded compatibility attempt without silently switching models.